In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy.special
import time


def deriv1(uVec, ikVec):
    #Compute spatial derivative in Fourier Space
    uhat = np.fft.rfft(uVec)
    uhat = uhat * ikVec
    return np.fft.irfft(uhat, n=len(uVec))


def kdvrhs(u, ikVec):
    #Solve nonlinear part of the KdV equation

    ux = deriv1(u, ikVec)
    return -u * ux


def rk4_nonlinear(u, kVec, dt):
    #Advance time step using RK4
    dt2 = 0.5 * dt
    dt6 = dt / 6

    k1 = kdvrhs(u, kVec)
    k2 = kdvrhs(u + dt2 * k1, kVec)
    k3 = kdvrhs(u + dt2 * k2, kVec)
    k4 = kdvrhs(u + dt * k3, kVec)

    u = u + dt6 * (k1 + 2.0 * (k2 + k3) + k4)
    return u


def linear_kdv_solver(u, ikVec3, dt):
    #Solve linear KdV equation
    uhat = np.fft.rfft(u)
    uhat = uhat * np.exp(ikVec3 * dt)
    return np.fft.irfft(uhat, n=len(u))


def OSfirst(u, dt, Q, ikVec, ikVec3):
    #First Order Operator Splitting
    for _ in range(Q):
        u = linear_kdv_solver(u, ikVec3, dt)
        u = rk4_nonlinear(u, ikVec, dt)
    return u


def OSsecond(u, dt, Q, ikVec, ikVec3):
    #Second Order Splitting
    u = linear_kdv_solver(u, ikVec3, 0.5 * dt)
    u = rk4_nonlinear(u, ikVec, dt)

    for _ in range(Q - 1):
        u = linear_kdv_solver(u, ikVec3, dt)
        u = rk4_nonlinear(u, ikVec, dt)

    u = linear_kdv_solver(u, ikVec3, 0.5 * dt)
    return u


def OSfourth(u, dt, Q, ikVec, ikVec3):
    #Yoshida's Fourth Order Operator Splitting
    w1 = 1.3512071919596576
    w0 = -1.7024143839193153

    dt2 = 0.5 * dt

    for _ in range(Q):

        # Stage 1
        u = linear_kdv_solver(u, ikVec3, w1 * dt2)
        u = rk4_nonlinear(u, ikVec, w1 * dt)

        # Stage 2
        u = linear_kdv_solver(u, ikVec3, (w1 + w0) * dt2)
        u = rk4_nonlinear(u, ikVec, w0 * dt)

        # Stage 3
        u = linear_kdv_solver(u, ikVec3, (w1 + w0) * dt2)
        u = rk4_nonlinear(u, ikVec, w1 * dt)

        # Final linear step
        u = linear_kdv_solver(u, ikVec3, w1 * dt2)

    return u


def OSsixth(u, dt, Q, ikVec, ikVec3):
    #Yoshida's Sixth Order Operator Splitting
    w3 = 0.784513610477560
    w2 = 0.235573213359357
    w1 = -1.17767998417887
    w0 = 1.0 - 2.0 * (w1 + w2 + w3)

    dt2 = 0.5 * dt

    # Initial half linear step
    u = linear_kdv_solver(u, ikVec3, w3 * dt2)

    for _ in range(Q - 1):

        u = rk4_nonlinear(u, ikVec, w3 * dt)
        u = linear_kdv_solver(u, ikVec3, (w3 + w2) * dt2)

        u = rk4_nonlinear(u, ikVec, w2 * dt)
        u = linear_kdv_solver(u, ikVec3, (w2 + w1) * dt2)

        u = rk4_nonlinear(u, ikVec, w1 * dt)
        u = linear_kdv_solver(u, ikVec3, (w1 + w0) * dt2)

        u = rk4_nonlinear(u, ikVec, w0 * dt)
        u = linear_kdv_solver(u, ikVec3, (w0 + w1) * dt2)

        u = rk4_nonlinear(u, ikVec, w1 * dt)
        u = linear_kdv_solver(u, ikVec3, (w1 + w2) * dt2)

        u = rk4_nonlinear(u, ikVec, w2 * dt)
        u = linear_kdv_solver(u, ikVec3, (w2 + w3) * dt2)

        u = rk4_nonlinear(u, ikVec, w3 * dt)
        u = linear_kdv_solver(u, ikVec3, w3 * dt)

    # Final stage
    u = rk4_nonlinear(u, ikVec, w3 * dt)
    u = linear_kdv_solver(u, ikVec3, (w3 + w2) * dt2)

    u = rk4_nonlinear(u, ikVec, w2 * dt)
    u = linear_kdv_solver(u, ikVec3, (w2 + w1) * dt2)

    u = rk4_nonlinear(u, ikVec, w1 * dt)
    u = linear_kdv_solver(u, ikVec3, (w1 + w0) * dt2)

    u = rk4_nonlinear(u, ikVec, w0 * dt)
    u = linear_kdv_solver(u, ikVec3, (w0 + w1) * dt2)

    u = rk4_nonlinear(u, ikVec, w1 * dt)
    u = linear_kdv_solver(u, ikVec3, (w1 + w2) * dt2)

    u = rk4_nonlinear(u, ikVec, w2 * dt)
    u = linear_kdv_solver(u, ikVec3, (w2 + w3) * dt2)

    u = rk4_nonlinear(u, ikVec, w3 * dt)
    u = linear_kdv_solver(u, ikVec3, w3 * dt2)

    return u

In [2]:
start_time = time.perf_counter()

# Domain
m = 0.2**2
L = 4 * scipy.special.ellipk(m**2)

# Spatial info
N = 2**10
x = np.linspace(0, L, N, endpoint=False)

# Final time
tf = 20.0

# Array of time step sizes
dt_vals = np.array([0.08, 0.04, 0.02, 0.01])

# Error storage
errors1 = []
errors2 = []
errors4 = []
errors6 = []

# Exact solution
sn, cn, dn, ph = scipy.special.ellipj(x, m**2)
snf, cnf, dnf, phf = scipy.special.ellipj(x - (8*m**2 - 4)*tf, m**2)

u0 = 12 * m**2 * (cn**2)
uf = 12 * m**2 * (cnf**2)

# Wave numbers and their derivatives
kVec = 2 * np.pi * np.fft.rfftfreq(N, d=L/N)
ikVec = 1j * kVec
ikVec3 = 1j * kVec**3

# Loop over the testing dt vals
for dt in dt_vals:

    # Number of time steps
    Q = int(tf / dt)

    # Solve using different splitting methods
    u1 = OSfirst(u0.copy(), dt, Q, ikVec, ikVec3)
    u2 = OSsecond(u0.copy(), dt, Q, ikVec, ikVec3)
    u4 = OSfourth(u0.copy(), dt, Q, ikVec, ikVec3)
    u6 = OSsixth(u0.copy(), dt, Q, ikVec, ikVec3)

    # Compute the errors
    errors1.append(np.linalg.norm(u1 - uf))
    errors2.append(np.linalg.norm(u2 - uf))
    errors4.append(np.linalg.norm(u4 - uf))
    errors6.append(np.linalg.norm(u6 - uf))

# Error table
print("\nCombined Error Table")
print("dt      |   OS1 Error   Ratio  |   OS2 Error   Ratio  |   OS4 Error   Ratio  |   OS6 Error   Ratio")
print("-" * 110)

for i in range(len(dt_vals)):

    if i == 0:
        print(f"{dt_vals[i]:.4f}  | "
              f"{errors1[i]:.3e}       -    | "
              f"{errors2[i]:.3e}       -    | "
              f"{errors4[i]:.3e}       -    | "
              f"{errors6[i]:.3e}       -")
    else:
        r1 = errors1[i-1] / errors1[i]
        r2 = errors2[i-1] / errors2[i]
        r4 = errors4[i-1] / errors4[i]
        r6 = errors6[i-1] / errors6[i]

        print(f"{dt_vals[i]:.4f}  | "
              f"{errors1[i]:.3e}   {r1:6.2f}   | "
              f"{errors2[i]:.3e}   {r2:6.2f}   | "
              f"{errors4[i]:.3e}   {r4:6.2f}   | "
              f"{errors6[i]:.3e}   {r6:6.2f}")

end_time = time.perf_counter()
print(f"Run time: {end_time - start_time:.2f} seconds")


Combined Error Table
dt      |   OS1 Error   Ratio  |   OS2 Error   Ratio  |   OS4 Error   Ratio  |   OS6 Error   Ratio
--------------------------------------------------------------------------------------------------------------
0.0800  | 2.108e-04       -    | 8.661e-05       -    | 2.890e-05       -    | 3.478e-06       -
0.0400  | 8.334e-05     2.53   | 1.436e-05     6.03   | 1.874e-06    15.42   | 6.983e-08    49.80
0.0200  | 3.995e-05     2.09   | 3.301e-06     4.35   | 1.193e-07    15.70   | 1.156e-09    60.39
0.0100  | 1.978e-05     2.02   | 8.090e-07     4.08   | 7.496e-09    15.92   | 1.834e-11    63.05
Run time: 19.44 seconds


In [3]:
dt = 0.001
Q = 1000
dx = L/N

u_final = OSfirst(u0, dt, Q, ikVec, ikVec3)
u2_final = OSsecond(u0, dt, Q, ikVec, ikVec3)
u4_final = OSfourth(u0, dt, Q, ikVec, ikVec3)
u6_final = OSsixth(u0, dt, Q, ikVec, ikVec3)

def ux(u):
    # Compute spatial derivative
    return deriv1(u, ikVec)


# Conserved quantities KdV
def cq(u):
    #Compute conserved quantities
    ux_val = deriv1(u, ikVec)

    cq1 = dx * np.sum(u)
    cq2 = dx * np.sum(u**2)
    cq3 = dx * np.sum((1/6) * u**3 - 0.5 * (ux_val**2))

    return cq1, cq2, cq3


# Calculate initial condition CQs
cq1_0 = dx * np.sum(u0)
cq2_0 = dx * np.sum(u0**2)
cq3_0 = dx * np.sum((1/6) * u0**3 - 0.5 * (deriv1(u0, ikVec)**2))


def cq_error(u, name):
    # Compute error in conserved quantities


    ux_val = ux(u)

    cq1 = dx * np.sum(u)
    cq2 = dx * np.sum(u**2)
    cq3 = dx * np.sum((1/6) * u**3 - 0.5 * (ux_val**2))

    print(f"{name}")
    print("CQ1 error:", abs(cq1 - cq1_0))
    print("CQ2 error:", abs(cq2 - cq2_0))
    print("CQ3 error:", abs(cq3 - cq3_0))
    print()


# Compare conserved quantities
print("Conserved quantity differences:")

cq_error(u_final, "OS1")
cq_error(u2_final, "OS2")
cq_error(u4_final, "OS4")
cq_error(u6_final, "OS6")

Conserved quantity differences:
OS1
CQ1 error: 6.938893903907228e-18
CQ2 error: 3.436920886779049e-17
CQ3 error: 1.3466365609421382e-13

OS2
CQ1 error: 6.938893903907228e-18
CQ2 error: 3.469446951953614e-17
CQ3 error: 7.19910242530375e-17

OS4
CQ1 error: 6.938893903907228e-18
CQ2 error: 1.945058697438995e-16
CQ3 error: 3.8868647883605334e-16

OS6
CQ1 error: 6.938893903907228e-18
CQ2 error: 3.6418350973788094e-16
CQ3 error: 7.271743970860278e-16

